# Notebook 03: Binary Relevance Classification

**Task:** Predict `relevant` (yes/no) from CVE description.
**Input:** `dataset/cti_unified_clean.csv`
**Strategy:** Oversample yes × 20 to fix 30:1 imbalance.
**Output:** `models/classification/binary_relevance.joblib`

## Step 0 — Setup

In [1]:
import warnings, json, joblib, time
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH    = PROJECT_ROOT / 'dataset' / 'cti_unified_clean.csv'
CLS_DIR      = PROJECT_ROOT / 'models' / 'classification'
RES_DIR      = PROJECT_ROOT / 'results' / '03_binary'
CLS_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42
sns.set_theme(style='whitegrid')
print('Output:', CLS_DIR)

Output: d:\Projects\done\thesis\cyber_threat_classification_n_emerging_threat_detection\models\classification


## Step 1 — Load and Rebalance (1,670 × 20 = 33,400 yes + 33,400 no)

In [2]:
df = pd.read_csv(DATA_PATH)
print(f'Total: {len(df):,}')

bin_df = df[df['relevant'].isin(['yes', 'no'])].copy()
yes_df = bin_df[bin_df['relevant'] == 'yes']
no_df  = bin_df[bin_df['relevant'] == 'no']
print(f'yes: {len(yes_df):,}  no: {len(no_df):,}  ratio: {len(no_df)/len(yes_df):.1f}:1')

OVERSAMPLE = 20
n_target = len(yes_df) * OVERSAMPLE
print(f'\nOversample yes × {OVERSAMPLE} = {n_target:,}')

yes_bal = yes_df.sample(n=n_target, replace=True, random_state=RANDOM_SEED)
no_bal  = no_df.sample(n=min(n_target, len(no_df)), replace=False, random_state=RANDOM_SEED)

balanced = pd.concat([yes_bal, no_bal]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
print(f'\nBalanced: {len(balanced):,}')
print(balanced['relevant'].value_counts().to_string())

Total: 52,136
yes: 1,670  no: 50,466  ratio: 30.2:1

Oversample yes × 20 = 33,400

Balanced: 66,800
relevant
yes    33400
no     33400


## Step 2 — Split

In [3]:
from sklearn.model_selection import train_test_split

X = balanced['sentence_text'].astype(str).tolist()
y = (balanced['relevant'] == 'yes').astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')
print(f'Pos rate train: {y_train.mean()*100:.1f}%')

Train: 53,440  Test: 13,360
Pos rate train: 50.0%


## Step 3 — Features

In [4]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

bow_bin = CountVectorizer(binary=True, max_features=15000, min_df=2)
X_bb_train = bow_bin.fit_transform(X_train)
X_bb_test  = bow_bin.transform(X_test)
print(f'BOW: {X_bb_train.shape}')

tfidf = TfidfVectorizer(max_features=20000, min_df=2, ngram_range=(1,2), sublinear_tf=True)
X_tfidf_train = tfidf.fit_transform(X_train)
X_tfidf_test  = tfidf.transform(X_test)
print(f'TF-IDF: {X_tfidf_train.shape}')

BOW: (53440, 15000)
TF-IDF: (53440, 20000)


In [ ]:
from sentence_transformers import SentenceTransformer
sbert = SentenceTransformer('all-MiniLM-L6-v2')
t0 = time.time()
X_sbert_train = sbert.encode(X_train, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
X_sbert_test  = sbert.encode(X_test,  batch_size=64, normalize_embeddings=True, show_progress_bar=True)
print(f'SBERT: {X_sbert_train.shape}  ({time.time()-t0:.1f}s)')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/835 [00:00<?, ?it/s]

## Step 4 — Train Base Models

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.special import expit

results = []

def fit_eval(name, model, X_tr, X_te, feat):
    t0 = time.time()
    model.fit(X_tr, y_train)
    elapsed = time.time() - t0
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_te)[:, 1]
    else:
        proba = expit(model.decision_function(X_te))
    pred = (proba >= 0.5).astype(int)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average='macro')
    auc = roc_auc_score(y_test, proba)
    print(f'{name:28s} acc={acc*100:.2f}% f1={f1*100:.2f}% auc={auc:.4f} ({elapsed:.1f}s)')
    results.append({'name': name, 'model': model, 'feature': feat,
                    'proba': proba, 'acc': acc, 'f1': f1, 'auc': auc})

def svm_cal(): return CalibratedClassifierCV(
    LinearSVC(class_weight='balanced', C=1.0, max_iter=5000, random_state=RANDOM_SEED), cv=3)

fit_eval('SVM+BOW-binary', svm_cal(), X_bb_train, X_bb_test, 'BOW-binary')
fit_eval('SVM+TF-IDF',     svm_cal(), X_tfidf_train, X_tfidf_test, 'TF-IDF')
fit_eval('NB+BOW-binary',  MultinomialNB(), X_bb_train, X_bb_test, 'BOW-binary')
fit_eval('NB+TF-IDF',      MultinomialNB(), X_tfidf_train, X_tfidf_test, 'TF-IDF')
fit_eval('XGB+TF-IDF',     XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                         eval_metric='logloss', random_state=RANDOM_SEED,
                                         n_jobs=-1, tree_method='hist'),
          X_tfidf_train, X_tfidf_test, 'TF-IDF')
fit_eval('LogReg+SBERT',   LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1, random_state=RANDOM_SEED),
          X_sbert_train, X_sbert_test, 'SBERT')
fit_eval('XGB+SBERT',      XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                         eval_metric='logloss', random_state=RANDOM_SEED,
                                         n_jobs=-1, tree_method='hist'),
          X_sbert_train, X_sbert_test, 'SBERT')

## Step 5 — Model Comparison

In [ ]:
res_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('model', 'proba')} for r in results])
res_df = res_df.sort_values('f1', ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(res_df))
ax.barh(x - 0.2, res_df['f1']*100, 0.4, label='Macro-F1', color='#2ca02c')
ax.barh(x + 0.2, res_df['acc']*100, 0.4, label='Accuracy', color='#1f77b4')
ax.set_yticks(x); ax.set_yticklabels(res_df['name']); ax.invert_yaxis()
ax.set_xlabel('Score (%)'); ax.legend()
ax.set_title('Binary — Individual Models')
plt.tight_layout(); plt.savefig(RES_DIR / '01_individual_models.png', dpi=120); plt.show()
print(res_df.round(4).to_string())

## Step 6 — Diversity Ensemble

In [ ]:
from scipy.stats import pearsonr

sem = [r for r in results if r['feature'] == 'SBERT']
lex = [r for r in results if r['feature'] != 'SBERT']
best_sem = max(sem, key=lambda r: r['f1'])
best_lex = max(lex, key=lambda r: r['f1'])
chosen = [best_lex, best_sem]
seen = {best_lex['feature'], best_sem['feature']}
print(f'Start: {best_lex["name"]} + {best_sem["name"]}')

def div_ok(c, chosen, rho=0.98):
    return all(abs(pearsonr(c['proba'], m['proba'])[0]) <= rho for m in chosen)

best_f1 = f1_score(y_test, (np.mean([m['proba'] for m in chosen], axis=0) >= 0.5).astype(int), average='macro')
for cand in sorted([r for r in results if r not in chosen], key=lambda r: -r['f1']):
    if cand['feature'] in seen or not div_ok(cand, chosen): continue
    trial = chosen + [cand]
    tproba = np.mean([m['proba'] for m in trial], axis=0)
    tf1 = f1_score(y_test, (tproba >= 0.5).astype(int), average='macro')
    if tf1 > best_f1:
        chosen = trial; seen.add(cand['feature']); best_f1 = tf1
        print(f'+ {cand["name"]} → F1={tf1*100:.2f}%')

print('\nFinal members:')
for r in chosen: print(f'  {r["name"]:28s} feat={r["feature"]:12s} f1={r["f1"]*100:.2f}%')

ens_proba = np.mean([m['proba'] for m in chosen], axis=0)
ens_pred  = (ens_proba >= 0.5).astype(int)
ens_acc = accuracy_score(y_test, ens_pred)
ens_f1  = f1_score(y_test, ens_pred, average='macro')
ens_auc = roc_auc_score(y_test, ens_proba)
print(f'\nENSEMBLE: acc={ens_acc*100:.2f}% f1={ens_f1*100:.2f}% auc={ens_auc:.4f}')

## Step 7 — Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
print(classification_report(y_test, ens_pred, target_names=['irrelevant', 'relevant'], digits=4))
cm = confusion_matrix(y_test, ens_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['irrelevant', 'relevant'], yticklabels=['irrelevant', 'relevant'])
plt.title(f'Binary Ensemble — Acc {ens_acc*100:.2f}% | F1 {ens_f1*100:.2f}%')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout(); plt.savefig(RES_DIR / '02_confusion_matrix.png', dpi=120); plt.show()

## Step 8 — 5-Fold CV

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_acc, cv_f1 = [], []
for fold, (tr, va) in enumerate(skf.split(X, y), 1):
    X_tr = [X[i] for i in tr]; X_va = [X[i] for i in va]
    y_tr = np.array([y[i] for i in tr]); y_va = np.array([y[i] for i in va])
    probas = []
    for m in chosen:
        mdl = clone(m['model'])
        if m['feature'] == 'BOW-binary':
            Xt = bow_bin.transform(X_tr); Xv = bow_bin.transform(X_va)
        elif m['feature'] == 'TF-IDF':
            Xt = tfidf.transform(X_tr); Xv = tfidf.transform(X_va)
        elif m['feature'] == 'SBERT':
            Xt = sbert.encode(X_tr, normalize_embeddings=True, show_progress_bar=False)
            Xv = sbert.encode(X_va, normalize_embeddings=True, show_progress_bar=False)
        else: continue
        mdl.fit(Xt, y_tr)
        probas.append(mdl.predict_proba(Xv)[:, 1])
    pred = (np.mean(probas, axis=0) >= 0.5).astype(int)
    cv_acc.append(accuracy_score(y_va, pred))
    cv_f1.append(f1_score(y_va, pred, average='macro'))
    print(f'Fold {fold}: acc={cv_acc[-1]*100:.2f}% f1={cv_f1[-1]*100:.2f}%')
print(f'\n5-fold CV: acc={np.mean(cv_acc)*100:.2f}% ± {np.std(cv_acc)*100:.2f}%')
print(f'           f1={np.mean(cv_f1)*100:.2f}% ± {np.std(cv_f1)*100:.2f}%')

## Step 9 — Save Bundle

In [ ]:
bundle = {
    'task': 'binary_relevance',
    'members': [{'name': r['name'], 'model': r['model'], 'feature': r['feature']} for r in chosen],
    'vectorizers': {'BOW-binary': bow_bin, 'TF-IDF': tfidf},
    'sbert_model_name': 'all-MiniLM-L6-v2',
    'labels': ['irrelevant', 'relevant'],
    'accuracy': round(float(ens_acc), 4),
    'macro_f1': round(float(ens_f1), 4),
    'roc_auc':  round(float(ens_auc), 4),
    'cv_5fold': {'acc_mean': round(float(np.mean(cv_acc)),4),
                 'acc_std':  round(float(np.std(cv_acc)),4),
                 'f1_mean':  round(float(np.mean(cv_f1)),4),
                 'f1_std':   round(float(np.std(cv_f1)),4)},
}
joblib.dump(bundle, CLS_DIR / 'binary_relevance.joblib')
print(f'✅ {CLS_DIR / "binary_relevance.joblib"}')
with open(CLS_DIR / 'binary_relevance_config.json', 'w') as f:
    json.dump({k: v for k, v in bundle.items() if k not in ('members','vectorizers')}, f, indent=2)
print('✅ config saved')

## Step 10 — Inference Demo

In [ ]:
samples = [
    'Remote code execution in Chrome allows attacker to execute arbitrary code.',
    'Heap buffer overflow in OpenSSL allows denial of service.',
    'SQL injection in admin panel allows authentication bypass.',
    'The weather today is sunny with a high of 75 degrees.',
]
def predict_binary(sentences, bundle, sbert_model):
    probas = []
    for m in bundle['members']:
        feat = m['feature']
        if feat == 'SBERT':
            X = sbert_model.encode(sentences, normalize_embeddings=True)
        else:
            X = bundle['vectorizers'][feat].transform(sentences)
        probas.append(m['model'].predict_proba(X)[:, 1])
    return np.mean(probas, axis=0)

probs = predict_binary(samples, bundle, sbert)
for s, p in zip(samples, probs):
    label = 'RELEVANT' if p >= 0.5 else 'irrelevant'
    print(f'[{label:10s}] conf={p:.3f} | {s[:60]}')

## Step 11 — Summary

In [ ]:
print('=' * 55)
print('BINARY CLASSIFICATION SUMMARY')
print('=' * 55)
print(f'Rebalanced rows : {len(balanced):,}')
print(f'Train / Test    : {len(X_train):,} / {len(X_test):,}')
print(f'Members         : {len(chosen)}')
for m in chosen: print(f'  - {m["name"]}')
print(f'\nTest accuracy   : {ens_acc*100:.2f}%')
print(f'Test macro-F1   : {ens_f1*100:.2f}%')
print(f'Test ROC-AUC    : {ens_auc:.4f}')
print(f'CV acc          : {np.mean(cv_acc)*100:.2f}% ± {np.std(cv_acc)*100:.2f}%')
print(f'CV F1           : {np.mean(cv_f1)*100:.2f}% ± {np.std(cv_f1)*100:.2f}%')